# FabricCare V3 · 252-feature Decision Tree experiment

Data source is [FabriCare_Dataset_V3](https://drive.google.com/drive/folders/1cHixSkWAvL3gK1Mhuw8s6G0E7RMYx0-8). Add its shortcut directly to **My Drive** under its published name. The setup verifies the V3 split manifest and stages its raw files locally. Run cells in order.

This feature experiment has its own export format. Use the MobileNetV2 Colab notebook for the app-compatible 72-feature Decision Tree and SVM exports.


In [ ]:
%pip -q install scikit-learn==1.6.1 scikit-image==0.26.0 opencv-python-headless==4.12.0.88 numpy==2.2.6 joblib==1.5.3

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from collections import Counter
import hashlib, json, platform, time
import cv2, numpy as np, sklearn, skimage, joblib
from skimage.feature import graycomatrix, graycoprops
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from tempfile import mkdtemp
drive.mount('/content/drive')

import csv
import hashlib
import io
from collections import Counter, defaultdict
from pathlib import Path

from PIL import Image, ImageOps

V3_URL = "https://drive.google.com/drive/folders/1cHixSkWAvL3gK1Mhuw8s6G0E7RMYx0-8"
V3_ROOT = Path("/content/drive/MyDrive/FabriCare_Dataset_V3")
MANIFEST_SHA256 = "f164042dcb4a0b4876fc5474532df1bb7b25403703e4a70852f19ada64c287ea"
SPLITS = ("train", "valid", "test")
CLASS_NAMES = ["cotton", "denim", "linen", "polyester", "silk", "wool"]
SEED = 42

def read_v3_manifest(root):
    manifest = root / "reports" / "split_manifest_70_20_10.csv"
    if not manifest.is_file():
        raise FileNotFoundError(f"Open {V3_URL} and add its shortcut to My Drive as {root.name}.")
    data = manifest.read_bytes()
    if hashlib.sha256(data).hexdigest() != MANIFEST_SHA256:
        raise ValueError("The split manifest differs from the audited V3 revision. Verify the folder and split file.")
    rows = list(csv.DictReader(io.StringIO(data.decode("utf-8-sig"), newline="")))
    counts, groups, digests = Counter(), defaultdict(set), set()
    for row in rows:
        split, label, filename = row["split"], row["fabric"], row["filename"]
        if split not in SPLITS or label not in CLASS_NAMES or not filename or Path(filename).name != filename:
            raise ValueError(f"Invalid V3 manifest entry: {row}")
        digest = row["sha256"]
        if digest in digests:
            raise ValueError(f"Duplicate content SHA-256 in the split manifest: {filename}")
        if not row["group_id"]:
            raise ValueError(f"Missing garment group: {filename}")
        digests.add(digest)
        groups[row["group_id"]].add(split)
        counts[split, label] += 1
    if any(len(value) > 1 for value in groups.values()):
        raise ValueError("A garment group appears in multiple splits.")
    for split in SPLITS:
        if any(counts[split, label] == 0 for label in CLASS_NAMES):
            raise ValueError(f"Missing fabric class in {split}.")
        print(split, {label: counts[split, label] for label in CLASS_NAMES})
    return rows

def stage_v3(root, records, target, *, padded):
    staged = []
    for record in records:
        label, split = record["fabric"], record["split"]
        source = root / "raw" / label / record["filename"]
        content = source.read_bytes()
        if hashlib.sha256(content).hexdigest() != record["sha256"]:
            raise ValueError(f"Image bytes differ from the V3 manifest: {source}")
        parent = target / split / label
        parent.mkdir(parents=True, exist_ok=True)
        if padded:
            destination = parent / (record["sha256"] + ".png")
            with Image.open(io.BytesIO(content)) as raw:
                # Normalize archival originals before submitting them to the classifier.
                rgb = ImageOps.exif_transpose(raw).convert("RGB")
                ImageOps.pad(rgb, (224, 224), method=Image.Resampling.BILINEAR,
                             color=(127, 127, 127)).save(destination)
        else:
            destination = parent / record["filename"]
            destination.write_bytes(content)
        staged.append((str(destination), CLASS_NAMES.index(label)))
    return staged

if sklearn.__version__ != '1.6.1':
    raise RuntimeError('Restart the runtime after package installation.')
DATASET_DIR = Path(mkdtemp(prefix='fabricare-v3-raw-', dir='/content'))
stage_v3(V3_ROOT, read_v3_manifest(V3_ROOT), DATASET_DIR, padded=False)
OUTPUT_DIR = Path('/content/drive/MyDrive/Fabricare_training_v3') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
print('Exports:', OUTPUT_DIR)


## Extract multiscale GLCM features and check duplicate images

For each image this computes gray, Lab color, and high-pass lightness GLCMs from the full center square and a tighter center crop. It yields 252 features. Same-label byte-identical decoded photos are retained once, preferring train, then valid, then test. A duplicate with conflicting labels stops the run.

In [ ]:
def glcm_features(image):
    if image is None:
        raise ValueError('Unreadable image')
    h, w = image.shape[:2]
    side = min(h, w)
    y, x = (h - side) // 2, (w - side) // 2
    square = image[y:y + side, x:x + side]
    regions = [cv2.resize(square, (96, 96), interpolation=cv2.INTER_AREA)]
    inset = side // 6
    crop = square[inset:side - inset, inset:side - inset]
    regions.append(cv2.resize(crop, (96, 96), interpolation=cv2.INTER_AREA))
    features = []
    for region in regions:
        lab = cv2.cvtColor(region, cv2.COLOR_BGR2LAB)
        gray = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY).astype(np.float32)
        light = lab[:, :, 0].astype(np.float32)
        residual = light - cv2.GaussianBlur(light, (0, 0), sigmaX=2)
        signals = [gray // 16, lab[:, :, 0] // 16, lab[:, :, 1] // 16,
                   lab[:, :, 2] // 16, np.clip((residual + 32) / 4, 0, 15)]
        for signal in signals:
            quantized = np.clip(signal, 0, 15).astype(np.uint8)
            matrix = graycomatrix(quantized, [1, 2, 4],
                [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
                levels=16, symmetric=True, normed=True)
            for prop in ['contrast', 'homogeneity', 'energy', 'correlation']:
                values = graycoprops(matrix, prop)
                features.extend(values.mean(axis=1))
                features.extend(np.ptp(values, axis=1))
        for channel in cv2.split(lab):
            features.extend([float(np.mean(channel)), float(np.std(channel))])
    return np.asarray(features, dtype=np.float32)

EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
data, counts, manifest, seen = {}, {}, [], {}
for split in ['train', 'valid', 'test']:
    split_dir = DATASET_DIR / split
    if not split_dir.is_dir():
        raise ValueError(f'Missing split directory: {split_dir}')
    x_rows, y_rows = [], []
    counts[split] = Counter()
    for label in CLASS_NAMES:
        class_dir = split_dir / label
        if not class_dir.is_dir():
            raise ValueError(f'Missing class directory: {class_dir}')
        for path in sorted(p for p in class_dir.rglob('*') if p.is_file() and p.suffix.lower() in EXTENSIONS):
            image = cv2.imread(str(path))
            if image is None:
                raise ValueError(f'Unreadable image: {path}')
            digest = hashlib.sha256(str(image.shape).encode() + image.tobytes()).hexdigest()
            previous = seen.get(digest)
            if previous:
                if previous['label'] != label:
                    raise ValueError(f'Identical image has conflicting labels: {previous["path"]} vs {path}')
                continue
            relative = str(path.relative_to(DATASET_DIR))
            seen[digest] = {'label': label, 'split': split, 'path': relative}
            x_rows.append(glcm_features(image))
            y_rows.append(label)
            counts[split][label] += 1
            manifest.append({'path': relative, 'split': split, 'label': label, 'decoded_sha256': digest})
    if not x_rows:
        raise ValueError(f'No usable photos in {split_dir}')
    data[split] = (np.vstack(x_rows), np.asarray(y_rows))
    print(split, data[split][0].shape, dict(counts[split]), flush=True)
assert all(data[s][0].shape[1] == 252 for s in data)
print('Retained unique images:', sum(map(len, [data[s][1] for s in data])))

## Select the tree on validation only

The search ranks candidates by validation accuracy, then macro F1 as a tie-breaker. No test results influence parameter selection.

In [ ]:
X_train, y_train = data['train']
X_valid, y_valid = data['valid']
trials = []
best_model, best_key, best_params = None, (-1.0, -1.0), None
for depth in [8, 12, 16, None]:
    for leaf in [2, 5, 10, 20]:
        for criterion in ['gini', 'entropy']:
            for class_weight in [None, 'balanced']:
                model = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf,
                    criterion=criterion, class_weight=class_weight, random_state=SEED)
                model.fit(X_train, y_train)
                prediction = model.predict(X_valid)
                accuracy = float(accuracy_score(y_valid, prediction))
                macro_f1 = float(f1_score(y_valid, prediction, labels=CLASS_NAMES, average='macro', zero_division=0))
                trial = {'max_depth': depth, 'min_samples_leaf': leaf, 'criterion': criterion,
                    'class_weight': class_weight, 'validation_accuracy': accuracy,
                    'validation_macro_f1': macro_f1}
                trials.append(trial)
                key = (accuracy, macro_f1)
                if key > best_key:
                    best_model, best_key, best_params = model, key, trial
print('Selected:', best_params)
print(classification_report(y_valid, best_model.predict(X_valid), labels=CLASS_NAMES, zero_division=0))

## Final held-out test evaluation

Run this once after selection. If test accuracy is below 80%, report that result as measured; do not continue tuning against this test split.

In [ ]:
X_test, y_test = data['test']
y_test_pred = best_model.predict(X_test)
test_accuracy = float(accuracy_score(y_test, y_test_pred))
test_macro_f1 = float(f1_score(y_test, y_test_pred, labels=CLASS_NAMES, average='macro', zero_division=0))
metrics = {
    'test_accuracy': test_accuracy,
    'target_accuracy': 0.80,
    'target_reached': test_accuracy >= 0.80,
    'test_macro_f1': test_macro_f1,
    'per_class': classification_report(y_test, y_test_pred, labels=CLASS_NAMES, output_dict=True, zero_division=0),
    'confusion_matrix_labels': CLASS_NAMES,
    'confusion_matrix': confusion_matrix(y_test, y_test_pred, labels=CLASS_NAMES).tolist(),
}
print(f'Test accuracy: {test_accuracy:.1%} (80% target: {"REACHED" if metrics["target_reached"] else "NOT REACHED"})')
print(f'Test macro F1: {test_macro_f1:.3f}')
print(classification_report(y_test, y_test_pred, labels=CLASS_NAMES, zero_division=0))
print('Confusion matrix (rows=actual, columns=predicted):\n', np.array(metrics['confusion_matrix']))

## Save the experiment to Drive

The exported tree expects the 252-feature extractor above. Fabricare's current API computes 74 legacy features; do not replace that deployed model with this file without updating and verifying the API feature extraction too.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
model_path = OUTPUT_DIR / 'fabric_glcm_decision_tree_252.joblib'
joblib.dump(best_model, model_path)
(OUTPUT_DIR / 'training_report.json').write_text(json.dumps({
    'dataset': str(DATASET_DIR), 'classes': CLASS_NAMES, 'feature_count': 252,
    'feature_description': 'multiscale gray/Lab/high-pass-lightness GLCM',
    'counts': {s: dict(counts[s]) for s in counts}, 'selected_parameters': best_params,
    'validation_trials': trials, 'evaluation': metrics,
    'target_accuracy': 0.80, 'target_reached': metrics['target_reached'],
    'python': platform.python_version(), 'scikit_learn': sklearn.__version__,
    'numpy': np.__version__, 'opencv': cv2.__version__, 'scikit_image': skimage.__version__,
    'joblib': joblib.__version__,
    'limitations': ['Exact decoded-image duplicates were removed; related photos of the same garment may remain across splits.',
        'This 252-feature tree is not compatible with the app API 74-feature extractor.'],
}, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'dataset_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
reloaded = joblib.load(model_path)
np.testing.assert_array_equal(reloaded.predict(X_test), best_model.predict(X_test))
print('Saved model and reports:', OUTPUT_DIR)